## Setup (run every session)
Large artifacts go to Google Drive (`PA3_OUT`); code and small results live in the repo clone. Use the **same GPU type** for every timed run and keep the tab open (Colab Pro has no background execution).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['PA3_OUT'] = '/content/drive/MyDrive/ATML-PA3/outputs'   # checkpoints, caches (persist)
os.environ['PA3_COCO'] = '/content/coco'                             # images on fast local disk
os.environ['HF_HOME'] = '/content/hf_cache'
REPO = 'https://github.com/adilawan1/ATML-PA3.git'                 # <-- set


In [ ]:
%cd /content
!test -d ATML-PA3 || git clone $REPO ATML-PA3
%cd /content/ATML-PA3
!git pull -q
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Task 3: Test-time compute
Put the supplied `ATML-PA3-Data` folder on Drive and point `PA3_T3_DATA` at it. Offline steps are CPU-only (a CPU runtime is enough); only the live run needs a GPU.

In [ ]:
os.environ['PA3_T3_DATA'] = '/content/drive/MyDrive/ATML-PA3-Data'   # <-- set
!python -m task3.data --dataset math --inspect | head -80

Check the printed columns against `configs/task3.yaml` (`columns:`) and the README, then build the pools.

In [ ]:
!python -m task3.data --dataset math
!python -m task3.data --dataset mathvista

### Offline: selection, signals, adaptive stopping

In [ ]:
for ds in ['math', 'mathvista']:
    !python -m task3.offline --dataset {ds}
    !python -m task3.signals --dataset {ds}
    !python -m task3.adaptive --dataset {ds}

### Live generation (GPU, FP16, batch size 1). Set `live.revision` in configs/task3.yaml from the README first.

In [ ]:
!python -m task3.live
!python -m task3.live --summarize

In [ ]:
!python -m task3.analyze

## Save small results back to the repo
Commit from Colab (needs a token) or download `results/` and commit locally.

In [ ]:
!git add results && git -c user.name=pa3 -c user.email=pa3@example.com commit -qm "results" || true
# !git push https://<token>@github.com/adilawan1/ATML-PA3.git HEAD